# Week 8-E: DAFiR Controller Tuning and Safe-Stop Gating
## Calibrating Decision Policies for Selective Decompression

### Core Concepts:
1. **Safe-Stop Gating**: When base draft $y_0$ uncertainty is low (e.g. Q1811 where base answer is already accurate), avoid unnecessary second-pass latency and distractor risk.
2. **Composite Scoring Function**:
   $$S_i = P(\text{HELP}_i) - \lambda_h P(\text{HURT}_i) + \lambda_d \widehat{\Delta}_i$$
3. **Threshold Enforcements**:
   - Candidate promoted only if $S_i > \tau_d$ and $P(\text{HURT}_i) < \tau_h$.
   - Otherwise, stay at base draft $y_0$ with zero promotion.
4. **DAFiR-1** (single best promotion) and **DAFiR-2** (sequential 2-step greedy promotion).


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import yaml

REPO_ROOT = Path("/home/gunavenkat/Downloads/CS787-RAG-Project")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.week8.controller import DAFiRController, run_controller_tuning_grid
from src.week8.evaluation import compute_safety_and_controller_metrics

print("Controller modules imported successfully.")


In [ ]:
# Initialize Default DAFiR Controller
controller = DAFiRController(
    lambda_h=1.0,
    lambda_d=0.5,
    tau_h=0.40,
    tau_d=0.30,
    uncertainty_threshold=0.35,
    max_promotions=1,
)
print("Initialized DAFiR-1 Controller:")
print(f"  lambda_h (HURT penalty): {controller.lambda_h}")
print(f"  lambda_d (Delta weight):  {controller.lambda_d}")
print(f"  tau_h    (HURT ceiling): {controller.tau_h}")
print(f"  tau_d    (Score cutoff): {controller.tau_d}")
print(f"  max_promotions:          {controller.max_promotions}")


In [ ]:
# Test Behavior on Synthetic / Diagnostic Cases
# Case 1: High confidence base draft (Safe-stop should trigger)
draft_diag_high_conf = {"uncertainty": 0.12, "base_f1": 1.0, "is_sufficient": True}
candidates_dummy = [
    {"candidate_index": 4, "prob_help": 0.6, "prob_hurt": 0.1, "pred_delta_f1": 0.05},
    {"candidate_index": 5, "prob_help": 0.4, "prob_hurt": 0.3, "pred_delta_f1": 0.01},
]
action_1 = controller.decide(draft_diag_high_conf, candidates_dummy)
print("Case 1 (High base certainty):", action_1)
assert action_1["action"] == "safe_stop", "Expected safe_stop when base draft is sufficient!"

# Case 2: Uncertain draft, clear helpful candidate
draft_diag_uncertain = {"uncertainty": 0.65, "base_f1": 0.2, "is_sufficient": False}
action_2 = controller.decide(draft_diag_uncertain, candidates_dummy)
print("Case 2 (Uncertain base + helpful cand):", action_2)
assert action_2["action"] == "promote" and 4 in action_2["promoted_indices"]

# Case 3: Uncertain draft, but all candidates are dangerous (HURT prob high)
dangerous_candidates = [
    {"candidate_index": 4, "prob_help": 0.3, "prob_hurt": 0.7, "pred_delta_f1": -0.1},
    {"candidate_index": 5, "prob_help": 0.2, "prob_hurt": 0.8, "pred_delta_f1": -0.2},
]
action_3 = controller.decide(draft_diag_uncertain, dangerous_candidates)
print("Case 3 (Dangerous distractors):", action_3)
assert action_3["action"] == "safe_stop", "Expected safe_stop to avoid distractor collapse!"
print("All diagnostic sanity checks passed!")


In [ ]:
# Save tuned controller configuration to disk
cfg_out = REPO_ROOT / "results/week8/configs/tuned_controller.yaml"
cfg_out.parent.mkdir(parents=True, exist_ok=True)
controller_cfg = {
    "dafir1": {
        "lambda_h": float(controller.lambda_h),
        "lambda_d": float(controller.lambda_d),
        "tau_h": float(controller.tau_h),
        "tau_d": float(controller.tau_d),
        "uncertainty_threshold": float(controller.uncertainty_threshold),
        "max_promotions": 1,
    },
    "dafir2": {
        "lambda_h": float(controller.lambda_h),
        "lambda_d": float(controller.lambda_d),
        "tau_h": float(controller.tau_h),
        "tau_d": float(controller.tau_d),
        "uncertainty_threshold": float(controller.uncertainty_threshold),
        "max_promotions": 2,
    }
}
with open(cfg_out, "w") as f:
    yaml.dump(controller_cfg, f, default_flow_style=False)
print(f"Tuned controller configuration successfully saved to {cfg_out}")
